In [1]:
import sys, os
import re
import gc
import glob
import pandas as pd
import ast
import itertools
import numpy as np
from pathlib import Path

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [3]:
from tqdm.notebook import tqdm

tqdm.pandas()

In [4]:
import warnings
warnings.filterwarnings("ignore")

### Loading package

In [5]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[1]
sys.path.append(str(repo_path))

In [6]:
from py.utils import verifyDir,verifyFile

In [7]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

('/media/felipe/DATA21/datasets/', '/media/felipe/DATA21/models/')

### Loading data

In [8]:
DATA_PATH=f"{cfg.DATA_PATH}crimebb/"
CSV_PATH = f"{DATA_PATH}/{cfg.YEAR}/csv/"
HF_PROCESSED = f"{DATA_PATH}/{cfg.YEAR}/csv/topics/{cfg.TOPIC_SEARCH}/"
FEATURES_PATH = f"{cfg.MODEL_PATH}crimebb/{cfg.YEAR}/creamskim/features/"

In [9]:
verifyDir(FEATURES_PATH)

### Settup variables

In [10]:
n_grams = (1,1) # (number of words, number of strides (jumps) )
num_words = 5000
min_df=0.01
max_df=0.99

dim_vec = 5000
min_count=10
window=15
sample=6e-5 
alpha=0.001
min_alpha=0.0007
negative=1

## Loading data

In [11]:
data_df = pd.read_csv(f"{HF_PROCESSED}/HackForums_Languages.csv", sep="\t", low_memory=False, encoding='utf-8')
data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067 entries, 0 to 1066
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   thread_id             1067 non-null   int64  
 1   annotations           1067 non-null   str    
 2   site_id               1067 non-null   int64  
 3   site_name             1067 non-null   str    
 4   board_id              1067 non-null   int64  
 5   board_title           1067 non-null   str    
 6   thread_title          1067 non-null   str    
 7   content               1067 non-null   str    
 8   cve_codes_list        1067 non-null   str    
 9   cve_unique_list       1067 non-null   str    
 10  r_IMG                 1067 non-null   str    
 11  r_CITING              1067 non-null   str    
 12  r_IFRAME              1067 non-null   str    
 13  r_LINK                1067 non-null   str    
 14  r_CODE                1067 non-null   str    
 15  r_ATTACHMENT          1067 non-n

#### Processing texts

In [12]:
data_df["lang_detected"].value_counts()

lang_detected
english       1050
not found       13
portuguese       3
russian          1
Name: count, dtype: int64

In [13]:
# LANGUAGE_TO_EVAL=[lang for lang in data_df["lang_detected"].unique() if lang!="not found"]
LANGUAGE_TO_EVAL=["english"]

In [14]:
filter_df = data_df[data_df["lang_detected"].isin(LANGUAGE_TO_EVAL)].copy()
filter_df.reset_index(inplace=True, drop=True)
filter_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   thread_id             1050 non-null   int64  
 1   annotations           1050 non-null   str    
 2   site_id               1050 non-null   int64  
 3   site_name             1050 non-null   str    
 4   board_id              1050 non-null   int64  
 5   board_title           1050 non-null   str    
 6   thread_title          1050 non-null   str    
 7   content               1050 non-null   str    
 8   cve_codes_list        1050 non-null   str    
 9   cve_unique_list       1050 non-null   str    
 10  r_IMG                 1050 non-null   str    
 11  r_CITING              1050 non-null   str    
 12  r_IFRAME              1050 non-null   str    
 13  r_LINK                1050 non-null   str    
 14  r_CODE                1050 non-null   str    
 15  r_ATTACHMENT          1050 non-n

In [15]:
from py.nlpToolkit import TextProcesser

txt_processer = TextProcesser(language_to_process="english", 
                              keyword_sep='famveer',
                              keep_emojis=True,
                              #exclude_pipe=['morphologizer', "parser", "ner", "attribute_ruler", "tagger"])
                              exclude_pipe=['tagger', 'parser', 'ner'])
print(txt_processer.get_nlp_pipeline())

([('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec object at 0x72c88612fef0>), ('text_filter', <py.nlpToolkit.nlpToolkit.text_processing.text_filter.TextFilter object at 0x72c88a100230>), ('attribute_ruler', <spacy.pipeline.attributeruler.AttributeRuler object at 0x72c885f43290>), ('lemmatizer', <spacy.lang.en.lemmatizer.EnglishLemmatizer object at 0x72c885f4c090>)], ['tok2vec', 'text_filter', 'attribute_ruler', 'lemmatizer'])


In [16]:
%%time
text_processed = txt_processer.process_texts_by_language(filter_df["content"].values)

Preprocessing texts:   0%|          | 0/1050 [00:00<?, ?it/s]

CPU times: user 2.94 s, sys: 253 ms, total: 3.19 s
Wall time: 22.4 s


In [17]:
filter_df["content_processed"] = text_processed

In [18]:
filter_df["content_length"] = filter_df["content"].apply(lambda x: len(x.split()) )

### Extracting Features

Text cleaning and processing:

* occurred in too many documents (max_df)
* occurred in too few documents (min_df)
* were cut off by feature selection (max_features).
* number of words to evaluate context (ngram)

In [19]:
# from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder()
# le.fit(topics_of_interest)
# le.transform(topics_of_interest)

### Dictionary

In [20]:
filter_df["annotations_class"] = filter_df["annotations"].astype('category').cat.codes

In [21]:
dict_features = {}
dict_features["content"] = filter_df["content"].values
dict_features["content_processed"] = filter_df["content_processed"].values
dict_features["annotations"] = filter_df["annotations"].values
dict_features["annotations_class"] = filter_df["annotations_class"].values

In [22]:
if cfg.PROCESS_TXT:
    documents = filter_df["content_processed"].values
    out_filename="_processed"
else:
    documents = filter_df["content"].values
    out_filename=""

In [23]:
text_input = [[word for word in doc.split()] for doc in documents]

In [24]:
# Create Dictionary
import gensim.corpora as corpora
id2word = corpora.Dictionary(text_input)

In [25]:
# Create Corpus: Term Document Frequency
corpus = [id2word.doc2bow(text) for text in text_input]

In [26]:
dict_features["corpus"] = corpus
dict_features["id2word"] = id2word

### TF-IDF

Term frequency (TF) = $\large \frac{\text{(Number of Occurrences of a word)}}{\text{(Total words in the document)}}$

Inverse Document Frequency (IDF) word = $\large \log(\frac{\text{(Total words in the document)}}{\text{(Number of documents containing the word))}}$

$TF-IDF_{score} = TF \times IDF$

In [27]:
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer

In [28]:
tf_idf = TfidfVectorizer(max_features=num_words, # features: most occurring words
                         ngram_range=n_grams, # len of words context
                         sublinear_tf=True, # sublinear tf scaler
                         min_df=min_df, # min_df: a word repeated at least in 5 docs
                         max_df=max_df, # max_df: only those words that occur in a maximum of 90% of all the documents
                        )

In [29]:
%%time
X_tfidf = tf_idf.fit_transform(documents).toarray()
X_tfidf.shape, X_tfidf.dtype

CPU times: user 77.8 ms, sys: 5.99 ms, total: 83.8 ms
Wall time: 83.1 ms


((1050, 1773), dtype('float64'))

In [30]:
X_tfidf[0][X_tfidf[0]>0]

array([0.45652709, 0.14578941, 0.24731878, 0.38199147, 0.28435155,
       0.53163925, 0.31293569, 0.31909178])

In [31]:
dict_features["X_tfidf"] = X_tfidf
dict_features["tfidf_names"] = tf_idf.get_feature_names_out()

### Word2Vec - Bag of Words

In [32]:
from sklearn.feature_extraction.text import CountVectorizer

In [33]:
cbow = CountVectorizer(max_features=num_words, # features: most occurring words
                       ngram_range=n_grams, # len of words context
                       min_df=min_df, # min_df: a word repeated at least in 5 docs
                       max_df=max_df, # max_df: only those words that occur in a maximum of 90% of all the documents
                      )

In [34]:
%%time
X_cbow = cbow.fit_transform(documents).toarray()
X_cbow.shape, X_cbow.dtype

CPU times: user 84.6 ms, sys: 3.98 ms, total: 88.5 ms
Wall time: 87.7 ms


((1050, 1773), dtype('int64'))

In [35]:
X_cbow[0][X_cbow[0]>0]

array([1, 1, 1, 1, 1, 1, 1, 1])

In [36]:
dict_features["X_cbow"] = X_cbow
dict_features["cbow_names"] = cbow.get_feature_names_out()

### Word2Vec - Skip Gram

In [37]:
from gensim.models.phrases import Phrases

In [38]:
word_list_per_doc = [row.split() for row in documents]

In [39]:
phrases = Phrases(word_list_per_doc, min_count=30, progress_per=10000)

In [40]:
from gensim.models.phrases import Phraser

In [41]:
bigram = Phraser(phrases)

In [42]:
sentences = bigram[word_list_per_doc]

In [43]:
from gensim.models import Word2Vec

In [44]:
skip_gram = Word2Vec(min_count=20, # word frequency
                      vector_size=dim_vec, #output vector
                      window=15, # distance between words
                      sg=1, #Skip-Gram
                      sample=6e-5, 
                      alpha=0.03, 
                      min_alpha=0.0007, 
                      negative=20)

In [45]:
skip_gram.build_vocab(sentences, progress_per=10000)

In [46]:
skip_gram.train(sentences, total_examples=skip_gram.corpus_count, epochs=skip_gram.epochs)

(385744, 1327390)

In [47]:
skip_gram.wv.most_similar('vulnerability', topn=20)

[('exploited', 0.9602066278457642),
 ('affected', 0.9488334655761719),
 ('affects', 0.9388108253479004),
 ('attackers', 0.9367966651916504),
 ('flaw', 0.92927485704422),
 ('flaws', 0.9255266189575195),
 ('advisory', 0.92214035987854),
 ('discovered', 0.9209315776824951),
 ('bug', 0.9182037711143494),
 ('critical', 0.9151310920715332),
 ('aware', 0.914844810962677),
 ('execute_arbitrary', 0.9132770895957947),
 ('lead', 0.9117562770843506),
 ('remotely', 0.910891592502594),
 ('patch', 0.9088729023933411),
 ('researchers', 0.908153235912323),
 ('rce', 0.907457172870636),
 ('special', 0.9052531123161316),
 ('reported', 0.9048716425895691),
 ('remote_code', 0.9035990238189697)]

In [48]:
skip_gram.wv.most_similar('exploit', topn=20)

[('silent', 0.925832986831665),
 ('code', 0.9162538051605225),
 ('unique_powershell', 0.9012954831123352),
 ('password_generator', 0.9009715914726257),
 ('cve', 0.8999466300010681),
 ('tips_viewer', 0.8998416662216187),
 ('en_de', 0.8996168971061707),
 ('account_manager', 0.8993083238601685),
 ('javascript_picture', 0.8979612588882446),
 ('generator_trillium', 0.8975232243537903),
 ('javascript', 0.8966447710990906),
 ('silent_javascript', 0.8945982456207275),
 ('unique_shellcode', 0.8945139646530151),
 ('silent_encoded', 0.8941051959991455),
 ('excel_ole', 0.8937397599220276),
 ('word_ole', 0.891803503036499),
 ('virtual_keyboard', 0.8911035060882568),
 ('silent_shortcut', 0.8859123587608337),
 ('v6_edge', 0.8851091265678406),
 ('excel_properties', 0.8841584920883179)]

In [49]:
skip_gram.wv.most_similar('fud', topn=20)

[('kapo', 0.9771274924278259),
 ('vouch', 0.9693294167518616),
 ('bought', 0.9601921439170837),
 ('protected_view', 0.9562496542930603),
 ('undetected', 0.9550182223320007),
 ('refud', 0.9527249336242676),
 ('buy', 0.9512952566146851),
 ('sir', 0.9498480558395386),
 ('dude', 0.9498478770256042),
 ('jdb', 0.9466438293457031),
 ('seller', 0.9419400691986084),
 ('macro', 0.9417415261268616),
 ('price', 0.9403558373451233),
 ('vouches', 0.9400766491889954),
 ('sales', 0.9390575885772705),
 ('paid', 0.9382573366165161),
 ('bro', 0.9340627789497375),
 ('saying', 0.9332529306411743),
 ('gmail', 0.9330230951309204),
 ('add_skype', 0.9288941621780396)]

In [50]:
skip_gram.wv.most_similar("cve", topn=20)

[('exploit', 0.8999467492103577),
 ('office', 0.8842051029205322),
 ('security', 0.8838372230529785),
 ('avaible_teamviewer', 0.8828741908073425),
 ('silent_doc', 0.8792058825492859),
 ('v6', 0.8751890063285828),
 ('word_ooxml', 0.8747665882110596),
 ('multisploit_tool', 0.8744704723358154),
 ('pdf', 0.8708648681640625),
 ('silent_shortcut', 0.8685734868049622),
 ('account_manager', 0.8683314323425293),
 ('exploits_fud', 0.8674173355102539),
 ('macro_pdf', 0.8654124736785889),
 ('v6_edge', 0.8608205318450928),
 ('trillium', 0.8604835867881775),
 ('en_de', 0.8592398166656494),
 ('id_update', 0.8591643571853638),
 ('silent', 0.8591465950012207),
 ('javascript_picture', 0.859048068523407),
 ('videos_videos', 0.8581790924072266)]

In [51]:
skip_gram.wv.most_similar("poc", topn=20)

[('knowledge', 0.9123270511627197),
 ('basically', 0.9103154540061951),
 ('exploitable', 0.9053804278373718),
 ('trying', 0.9048064351081848),
 ('believe', 0.9035171866416931),
 ('question', 0.9021221995353699),
 ('research', 0.901607871055603),
 ('thought', 0.9012280106544495),
 ('aren', 0.9008768796920776),
 ('paste', 0.8986960053443909),
 ('easier', 0.8976967334747314),
 ('cves', 0.896230936050415),
 ('important', 0.8930699825286865),
 ('doubt', 0.8913280963897705),
 ('nessus', 0.8908162117004395),
 ('cool', 0.8897343873977661),
 ('existing', 0.8881546854972839),
 ('actually', 0.8879321217536926),
 ('suggest', 0.88681560754776),
 ('wouldn', 0.8837267756462097)]

### Doc2Vec

In [52]:
from gensim.test.utils import common_texts
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

In [53]:
sentences_doc = [TaggedDocument(doc, [i]) for i, doc in enumerate(documents)]

In [54]:
doc2vec = Doc2Vec(vector_size=dim_vec, # output vector
                  min_count=min_count, # word frequency
                  window=window, # distance between words
                  sample=sample, # words with frequency lower than this value, are ignored
                  alpha=alpha, 
                  min_alpha=min_alpha, 
                  negative=negative # noisy words
                 )

In [55]:
doc2vec.build_vocab(sentences_doc, progress_per=10000)

In [56]:
doc2vec.train(sentences_doc, total_examples=doc2vec.corpus_count, epochs=doc2vec.epochs)

In [57]:
X_doc2vec = np.array([doc2vec.infer_vector([doc]) for doc in documents])
X_doc2vec.shape, X_doc2vec.dtype

((1050, 5000), dtype('float32'))

In [58]:
dict_features["X_doc2vec"] = X_doc2vec

### Saving Files

In [59]:
for k in dict_features.keys():
    print(k, len(dict_features[k]))

content 1050
content_processed 1050
annotations 1050
annotations_class 1050
corpus 1050
id2word 18237
X_tfidf 1050
tfidf_names 1773
X_cbow 1050
cbow_names 1773
X_doc2vec 1050


In [60]:
import pickle
# create a binary pickle file 
file_ = open(f"{FEATURES_PATH}metadata{out_filename}.pkl","wb")

# write the python object (dict) to pickle file
pickle.dump(dict_features, file_)

# close file
file_.close()